## Weeks 4–5 – Data Cleaning and Preparation 
Raw MLS data contains formatting inconsistencies, missing values, and fields that need transformation 
before analysis. This phase prepares the dataset for reliable analytics. 
Tasks 

• Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate, ContractStatusChangeDate) 

• Remove unnecessary or redundant columns 

• Handle missing values appropriately 

• Ensure numeric fields are properly typed 

• Remove or flag invalid numeric values: ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0, 
negative Bedrooms or Bathrooms 

In [1]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt

In [2]:
dtype_dict = {
    'ListAgentEmail': str,
    'BuyerAgencyCompensationType': str,
    'BuyerAgentAOR': str,
    'OriginatingSystemName': str,
    'latfilled': float,
    'lonfilled': float
}

df_listed = pd.read_csv('Week3_Enriched_Listed.csv')
df_sold = pd.read_csv('Week3_Enriched_Sold.csv')
display(df_listed.head(3))
print("listed\n",df_listed.shape,"\n",df_listed.columns)

display(df_sold.head(3))
print("sold\n",df_sold.shape,"\n",df_sold.columns)

C:\Users\wzy\AppData\Local\Temp\ipykernel_59480\1481545034.py:10: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  df_listed = pd.read_csv('Week3_Enriched_Listed.csv')
C:\Users\wzy\AppData\Local\Temp\ipykernel_59480\1481545034.py:11: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: ListAgentEmail, 3: FireplaceYN, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sold = pd.read_csv('Week3_Enriched_Sold.csv')


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,PricePerSqFt,ListYear,ListMonth,year_month,rate_30yr_fixed
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,NaN,NaN,90067,2105.0,177861.0,1029.976941,2024,1,2024-01,6.6425
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,3.0,Capistrano Unified,92677,254.0,5300.0,896.700143,2024,1,2024-01,6.6425
2,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,2.0,NaN,91108,NaN,9404.0,969.230769,2024,1,2024-01,6.6425


listed
 (567266, 65) 
 Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric',
       'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool',
       'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt',
       'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation',
       'BedroomsTotal', 'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrP

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,BuyerAgencyCompensation,latfilled,lonfilled,PricePerSqFt,SoldToListRatio,CloseYear,CloseMonth,PriceDiff,year_month,rate_30yr_fixed
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,False,499000.0,551985747,jwachter@cbnorcal.com,2024-01-26,240000.0,...,NaN,NaN,NaN,210.526316,0.813559,2024,1,-55000.0,2024-01,6.6425
1,SanDiego,SanDiego,NaN,False,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,815000.0,...,NaN,NaN,NaN,412.867275,1.072510,2024,1,55100.0,2024-01,6.6425
2,SanDiego,SanDiego,NaN,False,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,810000.0,...,NaN,NaN,NaN,410.334347,1.051948,2024,1,40000.0,2024-01,6.6425


sold
 (413494, 76) 
 Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric',
       'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool',
       'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric',
       'ListingId', 'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate

In [ ]:
#remove unnecessary or redundant columns
cols_to_drop = [
    'ListingId', 'ListingKeyNumeric', 'OriginatingSystemName', 
    'OriginatingSystemSubName', 'BuyerAgentMlsId', 'BuyerAgentAOR', 
    'ListAgentAOR', 'BuyerOfficeAOR',
    
    'ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName', 'ListAgentFullName', 
    'BuyerAgentFirstName', 'BuyerAgentLastName', 'CoListAgentFirstName', 'CoListAgentLastName', 
    'CoListOfficeName',
    
    'BuyerAgencyCompensationType', 'BuyerAgencyCompensation',
    
    'LotSizeArea', 'LotSizeSquareFeet', 'StreetNumberNumeric', 'UnparsedAddress'
]

df_listed = df_listed.drop(columns=cols_to_drop, errors='ignore')
df_sold = df_sold.drop(columns=cols_to_drop, errors='ignore')

# rename latfilled & lonfilled
if 'latfilled' in df_sold.columns and 'lonfilled' in df_sold.columns:
    df_sold = df_sold.drop(columns=['Latitude', 'Longitude'], errors='ignore')
    #need to use Google API to fill it
    df_sold = df_sold.rename(columns={'latfilled': 'Latitude', 'lonfilled': 'Longitude'})

In [4]:
date_cols = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']

df_listed[date_cols] = df_listed[date_cols].apply(pd.to_datetime, errors='coerce')
df_sold[date_cols] = df_sold[date_cols].apply(pd.to_datetime, errors='coerce')

print(df_listed[date_cols].dtypes)
print(df_sold[date_cols].dtypes)

CloseDate                   datetime64[us]
PurchaseContractDate        datetime64[us]
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
dtype: object
CloseDate                   datetime64[us]
PurchaseContractDate        datetime64[us]
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
dtype: object


In [5]:
# sold
initial_rows = len(df_sold)

# Remove or Flag Invalid Numeric Values
# Filter out negative DaysOnMarket, Bedrooms, and Bathrooms
df_sold = df_sold[df_sold['DaysOnMarket'] >= 0]
df_sold = df_sold[(df_sold['BedroomsTotal'] >= 0) | (df_sold['BedroomsTotal'].isnull())]
df_sold = df_sold[(df_sold['BathroomsTotalInteger'] >= 0) | (df_sold['BathroomsTotalInteger'].isnull())]

# Date Consistency Checks
# Validate the logical order: ListingContractDate -> PurchaseContractDate -> CloseDate
df_sold['listing_after_close_flag'] = df_sold['ListingContractDate'] > df_sold['CloseDate']
df_sold['purchase_after_close_flag'] = df_sold['PurchaseContractDate'] > df_sold['CloseDate']

# Mark overall timeline violations
df_sold['negative_timeline_flag'] = (
    df_sold['listing_after_close_flag'] | 
    df_sold['purchase_after_close_flag'] | 
    (df_sold['ListingContractDate'] > df_sold['PurchaseContractDate'])
)

# Geographic Data Checks
# Flag records with missing coordinates, sentinel null values (0), or implausible locations
df_sold['missing_coord_flag'] = df_sold['Latitude'].isnull() | df_sold['Longitude'].isnull()
df_sold['zero_coord_flag'] = (df_sold['Latitude'] == 0) | (df_sold['Longitude'] == 0)
df_sold['positive_longitude_flag'] = df_sold['Longitude'] > 0  # California coordinates must be negative

# Deliverable Summary Report
print("\n" + "="*15 + " Weeks 4-5 Data Audit Report " + "="*15)
print(f"Initial Row Count (Before numeric filters): {initial_rows}")
print(f"Cleaned Row Count: {len(df_sold)}")
print(f"Removed {initial_rows - len(df_sold)} records with invalid numeric values")

print("\n Date Consistency Flags:")
print(f" - Listing after Close (listing_after_close_flag): {df_sold['listing_after_close_flag'].sum()} records")
print(f" - Purchase after Close (purchase_after_close_flag): {df_sold['purchase_after_close_flag'].sum()} records")
print(f" - Overall Timeline Violation (negative_timeline_flag): {df_sold['negative_timeline_flag'].sum()} records")

print("\n Geographic Data Flags:")
print(f" - Missing Coordinates (missing_coord_flag): {df_sold['missing_coord_flag'].sum()} records")
print(f" - Zero Coordinates (zero_coord_flag): {df_sold['zero_coord_flag'].sum()} records")
print(f" - Implausible Longitude (positive_longitude_flag): {df_sold['positive_longitude_flag'].sum()} records")

# Export for Deliverable
df_sold.to_csv('Week5_Cleaned_Sold.csv', index=False)
print("success save new sold dataset in week 5.")


=============== Weeks 4-5 Data Audit Report ===============
Initial Row Count (Before numeric filters): 413494
Cleaned Row Count: 413447
Removed 47 records with invalid numeric values

 Date Consistency Flags:
 - Listing after Close (listing_after_close_flag): 61 records
 - Purchase after Close (purchase_after_close_flag): 240 records
 - Overall Timeline Violation (negative_timeline_flag): 508 records

 Geographic Data Flags:
 - Missing Coordinates (missing_coord_flag): 349637 records
 - Zero Coordinates (zero_coord_flag): 30397 records
 - Implausible Longitude (positive_longitude_flag): 33413 records
success save new sold dataset in week 5.


In [6]:
#listed
initial_listed_rows = len(df_listed)

# Remove Invalid Numeric Values for Listed Data
# check LivingArea, DOM, and Rooms.
#df_listed = df_listed[df_listed['LivingArea'] > 100]
df_listed = df_listed[df_listed['DaysOnMarket'] >= 0]
df_listed = df_listed[(df_listed['BedroomsTotal'] >= 0) | (df_listed['BedroomsTotal'].isnull())]
df_listed = df_listed[(df_listed['BathroomsTotalInteger'] >= 0) | (df_listed['BathroomsTotalInteger'].isnull())]

# Geographic Data Checks for Listed Data
df_listed['missing_coord_flag'] = df_listed['Latitude'].isnull() | df_listed['Longitude'].isnull()
df_listed['zero_coord_flag'] = (df_listed['Latitude'] == 0) | (df_listed['Longitude'] == 0)
df_listed['positive_longitude_flag'] = df_listed['Longitude'] > 0  # California coordinates must be negative

# Deliverable Summary Report for Listed Data
print("\n" + "="*15 + " Listed Data Audit Report " + "="*15)
print(f" Initial Listed Row Count: {initial_listed_rows}")
print(f" Cleaned Listed Row Count: {len(df_listed)}")
print(f" Removed {initial_listed_rows - len(df_listed)} records with invalid numeric values")

print("\n Geographic Data Flags (Listed Data):")
print(f" - Missing Coordinates: {df_listed['missing_coord_flag'].sum()} records")
print(f" - Zero Coordinates: {df_listed['zero_coord_flag'].sum()} records")
print(f" - Implausible Longitude: {df_listed['positive_longitude_flag'].sum()} records")

df_listed.to_csv('Week5_Cleaned_Listed.csv', index=False)
print("success save new listed dataset in week 5.")


=============== Listed Data Audit Report ===============
 Initial Listed Row Count: 567266
 Cleaned Listed Row Count: 567238
 Removed 28 records with invalid numeric values

 Geographic Data Flags (Listed Data):
 - Missing Coordinates: 80462 records
 - Zero Coordinates: 65 records
 - Implausible Longitude: 78 records
success save new listed dataset in week 5.


In [7]:
df_listed = pd.read_csv('Week5_Cleaned_Listed.csv')
df_sold = pd.read_csv('Week5_Cleaned_Sold.csv')

C:\Users\wzy\AppData\Local\Temp\ipykernel_59480\1480493238.py:2: DtypeWarning: Columns (0: Latitude, 1: Longitude) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sold = pd.read_csv('Week5_Cleaned_Sold.csv')


## Week 6

In [12]:

# Price Metrics
# Price Ratio (Close-to-List)
#df_sold['PriceRatio'] = df_sold['ClosePrice'] / df_sold['OriginalListPrice']????
df_sold['PriceRatio'] = df_sold['ClosePrice'] / df_sold['ListPrice']

# Price Per Square Foot (PPSF)
df_sold['PPSF'] = df_sold['ClosePrice'] / df_sold['LivingArea']

# Days on Market (raw field)

# Time/Cycle Metrics - Year / Month / YrMo. Derived from CloseDate 
# in datetime format (actually done in Weeks 4-5)
df_sold['PurchaseContractDate'] = pd.to_datetime(df_sold['PurchaseContractDate'], errors='coerce')
df_sold['ListingContractDate'] = pd.to_datetime(df_sold['ListingContractDate'], errors='coerce')
df_sold['CloseDate'] = pd.to_datetime(df_sold['CloseDate'], errors='coerce')

df_sold['CloseDate'] = pd.to_datetime(df_sold['CloseDate'], errors='coerce')
df_sold['Year'] = df_sold['CloseDate'].dt.year
df_sold['Month'] = df_sold['CloseDate'].dt.month
df_sold['YrMo'] = df_sold['CloseDate'].dt.to_period('M').astype(str)

# Close-to-Original-List Ratio (Captures the total discount from the very first asking price)
df_sold['CloseToOrigRatio'] = df_sold['ClosePrice'] / df_sold['OriginalListPrice']

# Using .dt.days to extract the integer number of days
# Listing-to-Contract Days (How long it took to find a buyer)
df_sold['ListingToContractDays'] = (df_sold['PurchaseContractDate'] - df_sold['ListingContractDate']).dt.days

# Contract to Close Days
df_sold['ContractToCloseDays'] = (df_sold['CloseDate'] - df_sold['PurchaseContractDate']).dt.days

# Recalculate DOM to ensure accuracy
df_sold['Calculated_DOM'] = (df_sold['CloseDate'] - df_sold['ListingContractDate']).dt.days

# Defensive Programming (Cleaning inf values)
metrics_cols = ['PPSF', 'PriceRatio', 'CloseToOrigRatio']
df_sold[metrics_cols] = df_sold[metrics_cols].replace([np.inf, -np.inf], np.nan)


# ***Segment Analysis (The Deliverables)
print("\n" + "="*15 + " Week 6 Deliverable: Segment Analysis " + "="*15)

# PropertyType and PropertySubType
print("\n--- Segment 1: Property Type Analysis ---")
prop_segment = df_sold.groupby(['PropertyType', 'PropertySubType']).agg({
    'ClosePrice': 'median',
    'PPSF': 'mean',
    'DaysOnMarket': 'median'
}).round(2)
display(prop_segment.head(10))

# CountyOrParish and MLSAreaMajor
print("\n--- Segment 2: Geographic Market Analysis ---")
geo_segment = df_sold.groupby(['CountyOrParish', 'MLSAreaMajor']).agg({
    'ClosePrice': 'median',
    'ListingToContractDays': 'median',
    'ContractToCloseDays': 'median'
}).round(2).sort_values(by='ClosePrice', ascending=False)
display(geo_segment.head(10))

# Competitive Intelligence (ListOffice vs BuyerOffice)
# aggregate by transaction count to find the top cooperating offices
print("\n--- Segment 3: Competitive Intelligence (Office Cooperation) ---")
office_segment = df_sold.groupby(['ListOfficeName', 'BuyerOfficeName']).size().reset_index(name='Transaction_Count')
office_segment = office_segment.sort_values(by='Transaction_Count', ascending=False)
display(office_segment.head(10))

print("\n Week 6 metrics engineered and segmented.")


=============== Week 6 Deliverable: Segment Analysis ===============

--- Segment 1: Property Type Analysis ---


ClosePrice     PPSF  DaysOnMarket
PropertyType PropertySubType                                      
Residential  BoatSlip              250000.0  1037.50          19.0
             Cabin                 236000.0   313.48          48.0
             CoOwnership           400000.0   485.71          24.0
             Condominium           628000.0   700.16          24.0
             DeededParking         577500.0   410.51          17.0
             Duplex                915000.0   639.28          21.0
             Farm                 1425000.0   898.66          16.0
             Loft                  687500.0   549.94          22.0
             ManufacturedHome      287500.0   199.84          42.5
             ManufacturedOnLand    325000.0   246.15          30.0


--- Segment 2: Geographic Market Analysis ---


ClosePrice  \
CountyOrParish MLSAreaMajor                                          
Kern           GLNV - Glennville                       450092500.0   
Orange         CR - Crystal Cove                        11618125.0   
               SH - Shady Canyon                        10140000.0   
Los Angeles    C32 - Malibu Beach                        9000000.0   
Santa Barbara  MCTO - Montecito                          7050000.0   
Los Angeles    HHIL - Hidden Hills                       6550000.0   
               144 - Manhattan Bch Hill                  6000000.0   
Orange         N9 - Lower Newport Bay - Balboa Island    5450000.0   
San Diego      92067 - Rancho Santa Fe                   4800000.0   
Orange         NP - Balboa Peninsula                     4650000.0   

                                                       ListingToContractDays  \
CountyOrParish MLSAreaMajor                                                    
Kern           GLNV - Glennville                                       223.5   
Orange         CR - Crystal Cove                                        39.0   
               SH - Shady Canyon                                        63.0   
Los Angeles    C32 - Malibu Beach                                       92.0   
Santa Barbara  MCTO - Montecito                                         62.0   
Los Angeles    HHIL - Hidden Hills                                      88.0   
               144 - Manhattan Bch Hill                                 67.5   
Orange         N9 - Lower Newport Bay - Balboa Island                   32.0   
San Diego      92067 - Rancho Santa Fe                                  44.0   
Orange         NP - Balboa Peninsula                                    49.5   

                                                       ContractToCloseDays  
CountyOrParish MLSAreaMajor                                                 
Kern           GLNV - Glennville                                      81.0  
Orange         CR - Crystal Cove                                      30.5  
               SH - Shady Canyon                                      39.5  
Los Angeles    C32 - Malibu Beach                                     21.0  
Santa Barbara  MCTO - Montecito                                       21.0  
Los Angeles    HHIL - Hidden Hills                                    37.0  
               144 - Manhattan Bch Hill                               33.0  
Orange         N9 - Lower Newport Bay - Balboa Island                 28.0  
San Diego      92067 - Rancho Santa Fe                                28.0  
Orange         NP - Balboa Peninsula                                  28.0


--- Segment 3: Competitive Intelligence (Office Cooperation) ---


,ListOfficeName,BuyerOfficeName,Transaction_Count
49347,Compass,Compass,7427
44013,Coldwell Banker Realty,Coldwell Banker Realty,3517
44036,Coldwell Banker Realty,Compass,1580
103345,Keller Williams Realty,Keller Williams Realty,1385
49318,Compass,Coldwell Banker Realty,1326
72315,First Team Real Estate,First Team Real Estate,1135
88697,Intero Real Estate Services,Intero Real Estate Services,861
16357,Berkshire Hathaway HomeServices California Pro...,Berkshire Hathaway HomeServices California Pro...,810
66744,Equity Union,Equity Union,753
157783,Real Broker,Real Broker,715



 Week 6 metrics engineered and segmented.
